<a href="https://colab.research.google.com/github/aliraza0321/neurofive-solutions-ML-internship/blob/main/Task%237%20Build%20a%20Proper%20ML%20Pipeline%20with%20Feature%20Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
# Import necessary libraries and modules for data manipulation, machine learning, and model persistence.
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import pickle

In [32]:
# Load the dataset from a CSV file into a pandas DataFrame.
df=pd.read_csv("Telco-Customer-Churn.csv")

# Display the number of rows and columns in the DataFrame.
print(df.shape)

# Display the first 5 rows of the DataFrame to get a glimpse of the data.
print(df.head())

# Display a concise summary of the DataFrame, including data types and non-null values.
print(df.info())

# Print all column names to easily inspect them.
print(df.columns)

(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

In [33]:
# Perform Exploratory Data Analysis (EDA) and preprocess data.
# Drop the 'customerID' column as it is a unique identifier and not useful for prediction.
df=df.drop(["customerID"],axis=1)

# Convert 'TotalCharges' column to numeric. 'errors='coerce'' will turn non-numeric values into NaN.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check for the number of NaN values created during the conversion.
print(df['TotalCharges'].isnull().sum())

# Fill any remaining NaN values in 'TotalCharges' with 0. This handles cases where charges might be genuinely zero (e.g., new customers).
df['TotalCharges'] = df['TotalCharges'].fillna(0)

11


In [34]:
# Handle categorical values, specifically the target variable 'Churn'.
# Map the 'Churn' column from 'Yes' and 'No' to numerical values 1 and 0, respectively.
df['Churn']=df['Churn'].map({'Yes':1,'No':0})

# Separate features (X) and target variable (y).
# X contains all columns except 'Churn'.
X=df.drop(['Churn'],axis=1)
# y contains only the 'Churn' column.
y=df['Churn']

# Identify categorical columns (object dtype) for one-hot encoding.
categorical=X.select_dtypes(include=['object']).columns
# Identify numerical columns (non-object dtype) for scaling.
numerical=X.select_dtypes(exclude=['object']).columns

In [35]:
# Create a ColumnTransformer to apply different preprocessing steps to different types of columns.
process=ColumnTransformer(
    transformers=[
        # 'cat': Apply OneHotEncoder to categorical columns to convert them into numerical format.
        # 'handle_unknown='ignore'' ensures that new, unseen categories in test data don't cause errors.
        ('cat',OneHotEncoder(handle_unknown='ignore'),categorical),
        # 'num': Apply StandardScaler to numerical columns to standardize their values.
        ('num',StandardScaler(),numerical)
])

# Split the dataset into training and testing sets.
# X and y are split with a 80% training size and 20% testing size.
# 'random_state=42' ensures reproducibility of the split.
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [36]:
final_pipeline=Pipeline(
    [
        # The first step in the pipeline is the 'process' (ColumnTransformer) defined earlier.
        # This handles both one-hot encoding of categorical features and standard scaling of numerical features.
        ('process',process),
        # The second step is the 'classifier', which is a Decision Tree Classifier in this case.
        ('classifier',DecisionTreeClassifier())
    ]
)

In [37]:
# Train the entire pipeline (preprocessing + classifier) using the training data.
# The 'fit' method applies the transformations and then trains the Decision Tree Classifier.
final_pipeline.fit(x_train,y_train)

# Make predictions on the test set using the trained pipeline.
y_pred=final_pipeline.predict(x_test)

# Calculate the accuracy of the model by comparing predicted labels with actual labels.
accuracy=accuracy_score(y_test,y_pred)

# Print the calculated accuracy.
print(accuracy)

0.7111426543647977


In [40]:
# Print the classification report, which includes precision, recall, f1-score, and support for each class.
print(classification_report(y_test,y_pred))

# Print the confusion matrix, which shows the counts of true positive, true negative, false positive, and false negative predictions.
print(confusion_matrix(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.81      0.79      0.80      1036
           1       0.46      0.48      0.47       373

    accuracy                           0.71      1409
   macro avg       0.63      0.64      0.63      1409
weighted avg       0.72      0.71      0.71      1409

[[823 213]
 [194 179]]


In [43]:
# Save the trained machine learning pipeline to a file using pickle.
# 'model.pkl' is the filename, and 'wb' opens the file in binary write mode.
with open('model.pkl','wb') as file:
  # Use pickle.dump to serialize the 'final_pipeline' object and save it to the file.
  pickle.dump(final_pipeline,file)

# Confirm that the model has been successfully saved.
print("Model is saved successfully using pickle")

Model is saved successfully using pickle
